In [0]:
%reload_ext autoreload
%autoreload 2
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

df_sales = spark.read.table("pyspark_real_time.silver.sales")
df_sales.show(5,truncate=False)

In [0]:
df_prod = spark.read.table("pyspark_real_time.bronze.products")
df_prod.show(5,truncate=False)

In [0]:
from utils.custom_utils import Transformations
from pyspark.sql.functions import *

# Read Silver sales
print("Reading silver table: pyspark_real_time.silver.sales")
df_sales = spark.read.table("pyspark_real_time.silver.sales")

# Read Bronze products
print("Reading bronze table: pyspark_real_time.bronze.products")
df_prod = spark.read.table("pyspark_real_time.bronze.products")

# Initialize transformation class
tf = Transformations(spark)

# Join sales and products
print("Joining sales and products...")
df_join = tf.sales_prod_join(df_sales, df_prod)

# Apply process timestamp
print("Applying process timestamp...")
df_join = tf.process_timestamp(df_join)

# Gold table name
gold_table = "pyspark_real_time.gold.sales_marts"

# Optional logging
if spark.catalog.tableExists(gold_table):
    print(f"Overwriting existing table: {gold_table}")
else:
    print(f"Creating new table: {gold_table}")

# Create or overwrite Gold table
(
    df_join.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(gold_table)
)

print(f"Gold table loaded successfully: {gold_table}")

## category_marts
# df_join.createOrReplaceTempView("sales_mart")
# category_marts = spark.sql("select category,sum(quantity * unit_price) as amount from sales_mart group by category order by amount desc")

category_marts = (
    df_join
    .groupBy("category")
    .agg(sum(col("quantity") * col("unit_price")).alias("amount"))
)

cat_marts_table = "pyspark_real_time.gold.category_marts"

# Optional logging
if spark.catalog.tableExists(cat_marts_table):
    print(f"Overwriting existing table: {cat_marts_table}")
else:
    print(f"Creating new table: {cat_marts_table}")


(
    category_marts.write
    .format("delta")    
    .mode("overwrite")
    .saveAsTable(cat_marts_table)
)

print(f"category marts table loaded successfully: {cat_marts_table}")

In [0]:
x = spark.read.table("pyspark_real_time.gold.sales_mart")
x.createOrReplaceTempView("sales_mart")
spark.sql("select category,sum(quantity * unit_price) as amount from sales_mart group by category order by amount desc").show(5,truncate=False)


In [0]:
spark.sql("DROP TABLE pyspark_real_time.gold.sales_mart")